In [1]:
# Covid-19 Data Analysis Capstone Project

# Step 1: Importing Libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Ignore warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Step 2: Load the Dataset (LOCAL LOAD FALLBACK)
# Check if file exists locally first
local_path = 'covid-data.csv'
if os.path.exists(local_path):
    try:
        df = pd.read_csv(local_path)
    except Exception as e:
        raise RuntimeError(f"Error reading the CSV file: {e}")
else:
    raise FileNotFoundError("Could not load dataset. Please download it manually from:\n"
                            "https://raw.githubusercontent.com/SR1608/Datasets/main/covid-data.csv\n"
                            "and save it as 'covid-data.csv' in the working directory.")

# Step 3: High-Level Data Understanding
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nInfo:")
df.info()
print("\nDescribe:\n", df.describe(include='all'))

# Step 4: Low-Level Data Understanding
print("\nUnique Locations:", df['location'].nunique())
print("\nContinent Value Counts:\n", df['continent'].value_counts(dropna=True))
print("\nMax Total Cases:", df['total_cases'].max())
print("Mean Total Cases:", df['total_cases'].mean())
print("\nTotal Deaths Quantiles:\n", df['total_deaths'].quantile([0.25, 0.5, 0.75]))

# Check for max/min safely
if not df['human_development_index'].dropna().empty:
    print("\nMax HDI Row:\n", df.loc[df['human_development_index'].idxmax()])
else:
    print("\nMax HDI Row: No valid data")

if not df['gdp_per_capita'].dropna().empty:
    print("\nMin GDP per Capita Row:\n", df.loc[df['gdp_per_capita'].idxmin()])
else:
    print("\nMin GDP per Capita Row: No valid data")

# Step 5: Filter Specific Columns
cols = ['continent', 'location', 'date', 'total_cases', 'total_deaths', 'gdp_per_capita', 'human_development_index']
df = df[cols].copy()

# Step 6: Data Cleaning
df.drop_duplicates(inplace=True)
print("\nMissing Values Before Cleaning:\n", df.isnull().sum())
df.dropna(subset=['continent'], inplace=True)
df.fillna(0, inplace=True)
print("\nMissing Values After Cleaning:\n", df.isnull().sum())

# Step 7: Date Conversion & Feature Extraction
try:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['month'] = df['date'].dt.month
except Exception as e:
    print(f"Error converting date column: {e}")

# Step 8: Aggregation by Continent
try:
    df_groupby = df.groupby('continent', as_index=False).max(numeric_only=True)
except Exception as e:
    print(f"Error during aggregation: {e}")

# Step 9: Feature Engineering
df_groupby['total_deaths_to_total_cases'] = df_groupby.apply(
    lambda row: row['total_deaths'] / row['total_cases'] if row['total_cases'] else 0, axis=1
)

# Step 10: Visualization

# a. GDP Histogram
try:
    plt.figure(figsize=(8, 4))
    sns.histplot(df['gdp_per_capita'], kde=True, bins=30, color='skyblue')
    plt.title('GDP Per Capita Distribution')
    plt.xlabel('GDP Per Capita')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()
except Exception as e:
    print(f"Error in histogram: {e}")

# b. Scatter Plot: Total Cases vs GDP
try:
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df, x='total_cases', y='gdp_per_capita', hue='continent')
    plt.title('Total Cases vs GDP Per Capita')
    plt.xlabel('Total Cases')
    plt.ylabel('GDP Per Capita')
    plt.grid(True)
    plt.show()
except Exception as e:
    print(f"Error in scatter plot: {e}")

# c. Pairplot
try:
    sns.pairplot(df_groupby)
    plt.suptitle("Pairplot of Aggregated Data", y=1.02)
    plt.show()
except Exception as e:
    print(f"Error in pairplot: {e}")

# d. Bar Plot: Total Cases by Continent
try:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_groupby, x='continent', y='total_cases', palette='viridis')
    plt.title('Total Cases by Continent')
    plt.xlabel('Continent')
    plt.ylabel('Total Cases')
    plt.grid(axis='y')
    plt.show()
except Exception as e:
    print(f"Error in bar plot: {e}")

# Step 11: Save Final Data
try:
    df_groupby.to_csv('df_groupby.csv', index=False)
    print("\nFinal summarized data saved as 'df_groupby.csv'")
except Exception as e:
    print(f"Error saving file: {e}")


ModuleNotFoundError: No module named 'pandas'